# 05 - Learned Multimodal Verifier (item 1)

Trains and evaluates the fusion classifier from `verifier_model.py`. Read that module's docstring and `load_gold_labels`'s docstring before trusting the gold-set size or comparison: n=70 gold claims (not the manuscript's n=149 for a differently-scoped population), and this session's local run found the rule-based / logistic-regression / fusion-MLP F1 confidence intervals overlap heavily -- no method is yet distinguishable from another at this sample size. Re-running here just reproduces that on Kaggle;
growing the gold set (finishing annotation of the full 500-instance stratified subset) is what would actually resolve it.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# verifier_model.main() resolves its own paths via a fixed project-root layout;
# call its pieces directly here against the Kaggle-resolved paths instead so it
# works both on Kaggle and locally without editing verifier_model.py.
import verifier_model as vm
import json
import numpy as np
from graph_utils import CitationGraph, load_relational_claims, verify
from hard_negatives import build_tfidf_index
from verifier_features import build_dataset, FEATURE_NAMES

graph = CitationGraph.load(REFS_CACHE)
all_claims = load_relational_claims(BENCH, CITE_RESULTS)

paper_abstracts = {}
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
tfidf_vectors = build_tfidf_index(paper_abstracts)

silver_labels = vm.load_silver_labels(CITE_RESULTS)
gold_labels = vm.load_gold_labels(BENCH_VERIFIED) if BENCH_VERIFIED else {}
silver_clean = {k: v for k, v in silver_labels.items() if k not in gold_labels}
print(f"Silver: {len(silver_clean)} | Gold: {len(gold_labels)}")

X_train, y_train, _ = build_dataset(all_claims, graph, tfidf_vectors, silver_clean)
X_gold, y_gold, gold_meta = build_dataset(all_claims, graph, tfidf_vectors, gold_labels)
print(f"Train: {X_train.shape} | Gold: {X_gold.shape}")

rule_preds = np.array([1 if verify(m["paper_a"], m["relation"], m["paper_b"], graph) == "supported" else 0
                       for m in gold_meta])
results = {"rule_based": vm.bootstrap_prf1(y_gold, rule_preds)}

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
logreg = LogisticRegression(max_iter=2000, class_weight="balanced").fit(scaler.transform(X_train), y_train)
results["logistic_regression"] = vm.bootstrap_prf1(y_gold, logreg.predict(scaler.transform(X_gold)))

print("Rule-based:", {k: v for k, v in results["rule_based"].items() if "ci" not in k})
print("Logistic regression:", {k: v for k, v in results["logistic_regression"].items() if "ci" not in k})

# -- Small MLP fusion classifier (matches verifier_model.py's FusionMLP shape) --
# Runs on CPU fine (just slower); the third arm of the 3-way comparison this
# notebook's own title promises -- was previously left as a "run this yourself"
# suggestion instead of actually run, which meant the notebook only produced 2
# of the 3 numbers cited in verifier_model.py's docstring.
import torch
import torch.nn as nn

torch.manual_seed(0)

class FusionMLP(nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 2),
        )

    def forward(self, x):
        return self.net(x)

Xtr_s, Xgold_s = scaler.transform(X_train), scaler.transform(X_gold)
Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.long)
Xgold_t = torch.tensor(Xgold_s, dtype=torch.float32)

# class-balanced loss weight, since silver "supported" heavily outnumbers "unsupported"
n_pos, n_neg = int(y_train.sum()), int((1 - y_train).sum())
class_weight = torch.tensor([n_pos / max(1, n_neg + n_pos), n_neg / max(1, n_neg + n_pos)], dtype=torch.float32)

mlp = FusionMLP(X_train.shape[1])
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weight)

n_epochs, batch_size, n_samples = 60, 256, Xtr_t.shape[0]
for epoch in range(n_epochs):
    perm = torch.randperm(n_samples)
    total_loss = 0.0
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i + batch_size]
        opt.zero_grad()
        out = mlp(Xtr_t[idx])
        loss = loss_fn(out, ytr_t[idx])
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(idx)
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f"  epoch {epoch}: train loss {total_loss / n_samples:.4f}")

mlp.eval()
with torch.no_grad():
    mlp_preds = mlp(Xgold_t).argmax(dim=1).numpy()
results["fusion_mlp"] = vm.bootstrap_prf1(y_gold, mlp_preds)
print("Fusion MLP:", {k: v for k, v in results["fusion_mlp"].items() if "ci" not in k})

# -- Paired comparisons: more powerful than eyeballing CI overlap above, since
# it's the same gold claims scored under every method -- see
# verifier_model.paired_bootstrap_diff's docstring for why this beats comparing
# two independently-computed CIs.
print("\nPaired bootstrap comparisons (positive = first method ahead):")
pairs = [("rule_vs_logreg", rule_preds, logreg.predict(scaler.transform(X_gold))),
         ("rule_vs_fusion_mlp", rule_preds, mlp_preds),
         ("logreg_vs_fusion_mlp", logreg.predict(scaler.transform(X_gold)), mlp_preds)]
for name, preds_a, preds_b in pairs:
    cmp = vm.paired_bootstrap_diff(y_gold, preds_a, preds_b)
    results[f"paired_{name}"] = cmp
    sig = " *SIGNIFICANT*" if cmp["f1_significant"] else ""
    print(f"  {name}: F1 diff={cmp['f1_diff']:+.3f} CI=[{cmp['f1_diff_ci'][0]:+.3f}, "
          f"{cmp['f1_diff_ci'][1]:+.3f}]{sig}")

with open(OUT_DIR / "verifier_model_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
torch.save(mlp.state_dict(), OUT_DIR / "fusion_mlp.pt")
print("Saved to", OUT_DIR / "verifier_model_results.json", "and", OUT_DIR / "fusion_mlp.pt")
